# Cloud model only — local robot simulator
Upload this notebook to Colab and select a GPU runtime. The simulator stays on your laptop. This notebook runs a LIBERO-Spatial OpenVLA checkpoint and exposes an authenticated temporary endpoint.

**Status:** endpoint protocol and mock integration are tested locally; GPU loading/inference and the public tunnel are not validated yet. Free GPU availability is not guaranteed. No paid resource is provisioned.

This downloads roughly 15 GB of checkpoint weights plus dependencies into the cloud runtime. Start with a 10-step local episode after the endpoint is ready. The 4-bit FP16-compute baseline differs from upstream BF16/FlashAttention evaluation; don't claim reproduction of published scores.


In [ ]:
import os, sys, subprocess, shutil, json, time, re
from pathlib import Path
import google.colab
if not Path('/content').exists():
    raise RuntimeError('Run in Colab; model weights must not be downloaded to the laptop.')
ROOT = Path('/content/vla-model')
ROOT.mkdir(exist_ok=True)
def run(args, **kwargs):
    return subprocess.run(args, check=True, text=True, **kwargs)
probe = run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader,nounits'], capture_output=True)
print(probe.stdout)
line = probe.stdout.splitlines()[0].split(',')
if float(line[1].strip()) < 12000:
    raise RuntimeError('This first model setup targets at least 12 GB VRAM; do not download weights yet.')
free = shutil.disk_usage(ROOT).free / 2**30
print('Free disk GiB:', round(free, 1))
if free < 30:
    raise RuntimeError('Need at least 30 GiB available before installing packages and checkpoint.')
(ROOT / 'hardware.json').write_text(json.dumps({'gpu':probe.stdout, 'free_disk_gib':free}, indent=2))


## Isolated Python 3.10 model environment
This avoids changing the notebook kernel's Python or installed packages. Run this once per fresh runtime. Package installation can take several minutes; save the full traceback if it fails.


In [ ]:
run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
UV = shutil.which('uv')
model_env = dict(os.environ, UV_CACHE_DIR=str(ROOT/'uv-cache'), UV_PYTHON_INSTALL_DIR=str(ROOT/'python'))
PYTHON = str(ROOT/'venv/bin/python')
if not Path(PYTHON).exists():
    run([UV, 'venv', '--python', '3.10', str(ROOT/'venv')], env=model_env)
run([UV, 'pip', 'install', '--python', PYTHON, 'torch==2.2.2', 'torchvision==0.17.2',
     '--index-url', 'https://download.pytorch.org/whl/cu121'], env=model_env)
(ROOT/"requirements.txt").write_text('# Install torch==2.2.2 and torchvision==0.17.2 from the cu121 index first.\nnumpy==1.26.4\npillow==10.4.0\ntransformers==4.40.1\ntokenizers==0.19.1\ntimm==0.9.10\naccelerate==0.30.1\nbitsandbytes==0.43.1\nhuggingface-hub==0.23.5\ntensorflow-cpu==2.15.1\nsentencepiece==0.1.99\n')
run([UV, 'pip', 'install', '--python', PYTHON, '-r', str(ROOT/'requirements.txt')], env=model_env)
freeze = run([UV, 'pip', 'freeze', '--python', PYTHON], env=model_env, capture_output=True).stdout
(ROOT/'requirements-resolved.txt').write_text(freeze)
run([PYTHON, '-c', 'import torch; print(torch.__version__, torch.cuda.is_available()); assert torch.cuda.is_available()'])


## Start the model endpoint
Enter a secret token of at least 16 ASCII letters/digits, and reuse it on the laptop. It is kept in process memory and is not printed. The server resolves and reports the model revision, loads 4-bit weights, and only becomes healthy after loading completes.

The server file is embedded below so you only need to upload this notebook. Changes to `cloud/policy_server.py` require regenerating this notebook.


In [ ]:
(ROOT/"policy_server.py").write_text('"""Model-only JSON endpoint. Run --mock for transport testing without weights."""\nimport argparse\nimport base64\nimport hmac\nimport io\nimport json\nimport os\nimport threading\nimport time\nfrom http.server import BaseHTTPRequestHandler, ThreadingHTTPServer\n\nimport numpy as np\nfrom PIL import Image\n\nCONTRACT = \'libero_osc_pose_v1\'\nCHECKPOINT = \'openvla/openvla-7b-finetuned-libero-spatial\'\n\n\ndef prepare_image(image, tf):\n    """Match the upstream LIBERO JPEG/Lanczos resize and 90%-area crop."""\n    encoded = tf.io.encode_jpeg(np.asarray(image, dtype=np.uint8))\n    decoded = tf.io.decode_jpeg(encoded, channels=3)\n    resized = tf.image.resize(decoded, [224, 224], method=\'lanczos3\', antialias=True)\n    quantized = tf.cast(tf.clip_by_value(tf.round(resized), 0, 255), tf.uint8)\n    offset = (1.0 - np.sqrt(0.9)) / 2.0\n    floating = tf.image.convert_image_dtype(quantized, tf.float32)[None]\n    cropped = tf.image.crop_and_resize(\n        floating, [[offset, offset, 1-offset, 1-offset]], [0], [224, 224])[0]\n    rgb = tf.image.convert_image_dtype(tf.clip_by_value(cropped, 0, 1), tf.uint8, saturate=True)\n    return Image.fromarray(rgb.numpy())\n\n\nclass Policy:\n    def __init__(self, mock=False, revision=\'main\', precision=\'4bit\'):\n        self.mock = mock\n        self.lock = threading.Lock()\n        self.metadata = {\'contract\': CONTRACT, \'suite\': \'libero_spatial\',\n                         \'policy\': \'mock\' if mock else CHECKPOINT,\n                         \'precision\': \'none\' if mock else precision,\n                         \'compute_dtype\': \'none\' if mock else \'float16\',\n                         \'quantization_type\': \'nf4\' if precision == \'4bit\' and not mock else None,\n                         \'attention\': \'none\' if mock else \'eager\',\n                         \'revision\': None, \'preprocessing\': \'none\' if mock else \'jpeg_lanczos224_crop_area0.9\',\n                         \'action_postprocessing\': \'gripper_binarize_and_invert\'}\n        if mock:\n            return\n        import torch\n        import tensorflow as tf\n        from huggingface_hub import model_info\n        from transformers import AutoProcessor, AutoModelForVision2Seq, BitsAndBytesConfig\n        if not torch.cuda.is_available():\n            raise RuntimeError(\'Cloud model needs an NVIDIA GPU. No CPU fallback or local weights.\')\n        tf.config.set_visible_devices([], \'GPU\')\n        resolved = model_info(CHECKPOINT, revision=revision).sha\n        self.metadata[\'revision\'] = resolved\n        options = dict(revision=resolved, trust_remote_code=True,\n                       torch_dtype=torch.float16, attn_implementation=\'eager\',\n                       low_cpu_mem_usage=True, device_map={\'\': 0})\n        if precision == \'4bit\':\n            options[\'quantization_config\'] = BitsAndBytesConfig(\n                load_in_4bit=True, bnb_4bit_quant_type=\'nf4\',\n                bnb_4bit_compute_dtype=torch.float16)\n        self.model = AutoModelForVision2Seq.from_pretrained(CHECKPOINT, **options).eval()\n        self.processor = AutoProcessor.from_pretrained(\n            CHECKPOINT, revision=resolved, trust_remote_code=True)\n        keys = self.model.norm_stats\n        self.unnorm_key = next((k for k in [\'libero_spatial\', \'libero_spatial_no_noops\'] if k in keys), None)\n        if self.unnorm_key is None:\n            raise RuntimeError(\'Checkpoint lacks LIBERO-Spatial action statistics.\')\n        self.metadata[\'unnorm_key\'] = self.unnorm_key\n        self.torch, self.tf = torch, tf\n\n    def predict(self, image, instruction):\n        if self.mock:\n            return [0, 0, 0, 0, 0, 0, -1]\n        with self.lock, self.torch.inference_mode():\n            image = prepare_image(image, self.tf)\n            prompt = \'In: What action should the robot take to \' + instruction.lower() + \'?\\nOut:\'\n            batch = self.processor(prompt, image).to(\'cuda:0\', dtype=self.torch.float16)\n            # Keep the mask aligned with OpenVLA\'s required terminal prompt token.\n            if batch[\'input_ids\'][0, -1].item() != 29871:\n                ids = batch[\'input_ids\']\n                batch[\'input_ids\'] = self.torch.cat([ids, ids.new_full((1, 1), 29871)], dim=1)\n                mask = batch[\'attention_mask\']\n                batch[\'attention_mask\'] = self.torch.cat([mask, mask.new_ones((1, 1))], dim=1)\n            action = np.asarray(self.model.predict_action(\n                **batch, unnorm_key=self.unnorm_key, do_sample=False), dtype=float).copy()\n            # Convert OpenVLA\'s [0,1] gripper convention to LIBERO\'s signed control.\n            action[-1] = -float(np.sign(2 * action[-1] - 1))\n            return action.tolist()\n\n\ndef make_server(policy, token, host=\'127.0.0.1\', port=8000):\n    if not token:\n        raise ValueError(\'Set VLA_API_TOKEN before starting the endpoint.\')\n\n    class Handler(BaseHTTPRequestHandler):\n        def log_message(self, fmt, *args):\n            pass  # Do not write tokens or observation payloads to access logs.\n\n        def send_json(self, status, payload):\n            body = json.dumps(payload, allow_nan=False).encode()\n            self.send_response(status)\n            self.send_header(\'Content-Type\', \'application/json\')\n            self.send_header(\'Content-Length\', str(len(body)))\n            self.end_headers()\n            self.wfile.write(body)\n\n        def authorized(self):\n            if not hmac.compare_digest(self.headers.get(\'Authorization\', \'\'), \'Bearer \' + token):\n                self.send_json(401, {\'error\': \'unauthorized\'})\n                return False\n            return True\n\n        def do_GET(self):\n            if self.authorized():\n                self.send_json(200 if self.path == \'/health\' else 404,\n                               policy.metadata if self.path == \'/health\' else {\'error\': \'not_found\'})\n\n        def do_POST(self):\n            if not self.authorized():\n                return\n            if self.path != \'/act\':\n                self.send_json(404, {\'error\': \'not_found\'})\n                return\n            try:\n                length = int(self.headers.get(\'Content-Length\', 0))\n                if not 0 < length <= 2_000_000:\n                    raise ValueError(\'Request must be at most 2 MB.\')\n                request = json.loads(self.rfile.read(length))\n                if not isinstance(request, dict):\n                    raise ValueError(\'Expected a JSON object.\')\n                if request.get(\'contract\') != CONTRACT:\n                    raise ValueError(\'Wrong action contract.\')\n                instruction = request[\'instruction\']\n                if not isinstance(instruction, str) or not 1 <= len(instruction) <= 2000:\n                    raise ValueError(\'Invalid instruction.\')\n                raw = base64.b64decode(request[\'image_png\'], validate=True)\n                with Image.open(io.BytesIO(raw)) as source:\n                    if source.size != (256, 256):\n                        raise ValueError(\'Expected rotated 256x256 RGB camera image.\')\n                    image = source.convert(\'RGB\')\n            except (ValueError, KeyError, TypeError, OSError) as exc:\n                self.send_json(400, {\'error\': str(exc)})\n                return\n            try:\n                start = time.perf_counter()\n                action = policy.predict(image, instruction)\n                values = np.asarray(action, dtype=float)\n                if values.shape != (7,) or not np.isfinite(values).all() or (np.abs(values) > 1.000001).any():\n                    raise ValueError(\'Policy returned invalid LIBERO control values.\')\n                self.send_json(200, {\'contract\': CONTRACT, \'action\': action,\n                                    \'inference_seconds\': time.perf_counter()-start})\n            except Exception:\n                import traceback\n                traceback.print_exc()\n                self.send_json(500, {\'error\': \'Policy inference failed; inspect cloud server log.\'})\n\n    return ThreadingHTTPServer((host, port), Handler)\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--mock\', action=\'store_true\')\n    parser.add_argument(\'--port\', type=int, default=8000)\n    parser.add_argument(\'--revision\', default=\'main\')\n    parser.add_argument(\'--precision\', choices=[\'4bit\', \'fp16\'], default=\'4bit\')\n    args = parser.parse_args()\n    token = os.environ.get(\'VLA_API_TOKEN\', \'\')\n    if not token:\n        parser.error(\'Set VLA_API_TOKEN.\')\n    policy = Policy(args.mock, args.revision, args.precision)\n    print(json.dumps(policy.metadata), flush=True)\n    make_server(policy, token, port=args.port).serve_forever()\n')
from getpass import getpass
TOKEN = getpass('Choose API token (at least 16 ASCII characters): ')
if len(TOKEN) < 16 or not TOKEN.isascii():
    raise ValueError('Use at least 16 ASCII characters.')
server_env = dict(os.environ, VLA_API_TOKEN=TOKEN, HF_HOME=str(ROOT/'hf-cache'), TF_CPP_MIN_LOG_LEVEL='2')
# Re-running this cell stops only the model server started by this notebook.
if 'server' in globals() and server.poll() is None:
    server.terminate()
    server.wait(timeout=30)
server_log = open(ROOT/'server.log', 'w')
server = subprocess.Popen([PYTHON, '-u', str(ROOT/'policy_server.py'), '--precision', '4bit'],
                          env=server_env, stdout=server_log, stderr=subprocess.STDOUT)
from urllib.request import Request, urlopen
HEALTH = None
for attempt in range(900):
    if server.poll() is not None:
        print((ROOT/'server.log').read_text()[-16000:])
        raise RuntimeError('Model server exited. Share the traceback above.')
    try:
        req = Request('http://127.0.0.1:8000/health', headers={'Authorization':'Bearer '+TOKEN})
        with urlopen(req, timeout=2) as response:
            HEALTH = json.load(response)
        break
    except OSError:
        if attempt % 10 == 0:
            print('Waiting for download/model loading...', attempt*2, 'seconds', flush=True)
        time.sleep(2)
if HEALTH is None:
    server.terminate()
    raise RuntimeError('Model did not become ready within 30 minutes. Inspect server.log.')
print(json.dumps(HEALTH, indent=2))
(ROOT/'model-metadata.json').write_text(json.dumps(HEALTH, indent=2))


## Open a temporary HTTPS endpoint
Cloudflare Quick Tunnels need no Cloudflare account, but have no uptime guarantee. The tunnel routes requests to the model process; all endpoint requests require your bearer token. Nothing from your laptop is uploaded until you launch the local client.


In [ ]:
import urllib.request
binary = ROOT/'cloudflared'
if not binary.exists():
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', binary)
    binary.chmod(0o755)
print(run([str(binary), '--version'], capture_output=True).stdout)
if 'tunnel' in globals() and tunnel.poll() is None:
    tunnel.terminate()
    tunnel.wait(timeout=30)
tunnel_log = open(ROOT/'tunnel.log', 'w')
tunnel = subprocess.Popen([str(binary), 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
                          stdout=tunnel_log, stderr=subprocess.STDOUT)
ENDPOINT = None
for _ in range(60):
    if tunnel.poll() is not None:
        raise RuntimeError((ROOT/'tunnel.log').read_text()[-6000:])
    matches = re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com', (ROOT/'tunnel.log').read_text())
    if matches:
        ENDPOINT = matches[-1]
        break
    time.sleep(1)
if ENDPOINT is None:
    raise RuntimeError('Tunnel URL not found. Inspect tunnel.log.')
print('Endpoint URL:', ENDPOINT)
print('On laptop: export VLA_ENDPOINT="'+ENDPOINT+'"')
print('Keep this runtime open. Use the same secret token on the laptop.')


## Run on your laptop
Open a terminal in the project folder. Copy the URL printed above into the first line:

```bash
export VLA_ENDPOINT="https://YOUR-URL.trycloudflare.com"
read -rsp 'API token: ' VLA_API_TOKEN
export VLA_API_TOKEN
MUJOCO_GL=glfw .venv-libero/bin/python scripts/run_remote_episode.py --steps 10
```

Enter the same token you chose above. This short run is an integration check, not a task-success benchmark. It saves video, actions, model metadata, and request timings in `results/remote-*`. If that works, a full LIBERO-Spatial episode uses `--steps 220`.

If you see an inference error, inspect the model log below. Never share your API token.


In [ ]:
print((ROOT/'server.log').read_text()[-16000:])


## Stop the endpoint when finished
This stops the two processes started here. The temporary URL then stops working. Colab may also disconnect or reclaim the free runtime.


In [ ]:
for name in ['tunnel', 'server']:
    process = globals().get(name)
    if process is not None and process.poll() is None:
        process.terminate()
        process.wait(timeout=30)
print('Model server and tunnel stopped.')


Sources: [OpenVLA](https://github.com/openvla/openvla), [LIBERO-Spatial checkpoint](https://huggingface.co/openvla/openvla-7b-finetuned-libero-spatial), [upstream image preprocessing](https://github.com/openvla/openvla/blob/main/experiments/robot/libero/libero_utils.py), [Quick Tunnels](https://developers.cloudflare.com/tunnel/get-started/quick-tunnels/).
